# POE - Bronze Layer

This notebook ingests the MIMIC-IV `poe` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm the table structure and the identifiers used to link provider orders back to patients and admissions.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/hosp/poe.csv"

poe = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {poe.count():,}")
print(f"Columns: {len(poe.columns)}")

poe.printSchema()

### Source Check

`POE` contains clinical orders entered for hospitalised patients.

The main order, patient, admission and provider identifiers are checked before ingestion so the key relationships can be understood and preserved in Bronze.

In [0]:
# Check key fields
key_columns = [
    "poe_id",
    "poe_seq",
    "subject_id",
    "hadm_id",
    "ordertime",
    "order_provider_id"
]

for column in key_columns:
    nulls = poe.filter(F.col(column).isNull()).count()
    print(f"{column}: {nulls:,} nulls")

# Check order identifier uniqueness
poe_id_unique = poe.select("poe_id").distinct().count()

poe_seq_unique = (
    poe
    .select("poe_id", "poe_seq")
    .distinct()
    .count()
)

print(f"\nUnique poe_id: {poe_id_unique:,}")
print(f"Unique poe_id + poe_seq: {poe_seq_unique:,}")
print(f"Total rows: {poe.count():,}")

### Source Check

`poe_id` is complete and unique, so it can be used as the main order identifier.

The patient, admission and order time fields are complete. Some orders do not have a provider ID, so those missing values are preserved for later Silver-layer validation.

In [0]:
# Add ingestion metadata
poe_bronze = (
    poe
    .withColumn("_source_file", F.lit("poe.csv"))
    .withColumn("_source_module", F.lit("hosp"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    poe_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.poe_raw")
)

print("Created: workspace.bronze.poe_raw")

In [0]:
source_count = poe.count()
bronze_count = spark.table("workspace.bronze.poe_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")

### Bronze Result

`POE` was successfully written to the Bronze layer with all source records preserved.

`poe_id` is complete and unique, while missing provider IDs have been kept unchanged for later Silver-layer validation.